# Template phân tích tự động (dùng với Papermill)

Notebook này đóng vai trò `SPSS_auto_template.ipynb` trong hướng dẫn Papermill.
Ô ngay dưới đây được gắn tag `parameters` — Papermill sẽ tự động ghi đè các biến
này khi chạy ngầm bằng `run_pipeline.py`.

Quy trình: nạp dữ liệu -> kiểm định giả định -> chạy kiểm định phù hợp -> vẽ đồ thị
-> lưu lại chính notebook này (đã chạy xong) làm bằng chứng kiểm toán (audit trail).

In [ ]:
# Các giá trị này sẽ bị Papermill ghi đè khi chạy tự động
file_du_lieu = "../sample_data/khao_sat_mau.csv"
muc_y_nghia = 0.05
bien_phu_thuoc = "ket_qua_hoc_tap"
bien_nhom = "nhom_thuc_nghiem"

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

df = pd.read_csv(file_du_lieu)
print(f"Đã nạp {len(df)} dòng từ {file_du_lieu}")
df.head()

## Bước 1 — Kiểm định giả định (tư duy chẩn đoán)

Không bao giờ chạy thẳng t-test/ANOVA mà không kiểm tra giả định trước — đúng như
cảnh báo trong slide "Rủi ro của thống kê nút bấm": nếu phương sai 2 nhóm không
đồng nhất, hệ thống phải TỰ ĐỘNG rẽ nhánh sang Welch's t-test/Mann-Whitney U.

In [ ]:
groups = df[bien_nhom].dropna().unique().tolist()
assert len(groups) == 2, f"Cần đúng 2 nhóm, hiện có: {groups}"
g1 = df.loc[df[bien_nhom] == groups[0], bien_phu_thuoc].dropna()
g2 = df.loc[df[bien_nhom] == groups[1], bien_phu_thuoc].dropna()

w1, p1 = stats.shapiro(g1)
w2, p2 = stats.shapiro(g2)
normal_ok = (p1 >= muc_y_nghia) and (p2 >= muc_y_nghia)

lev_stat, lev_p = stats.levene(g1, g2)
equal_var = lev_p >= muc_y_nghia

print(f"Shapiro-Wilk: p1={p1:.4f}, p2={p2:.4f} -> {'CHUẨN' if normal_ok else 'KHÔNG CHUẨN'}")
print(f"Levene: p={lev_p:.4f} -> phương sai {'ĐỒNG NHẤT' if equal_var else 'KHÔNG ĐỒNG NHẤT'}")

## Bước 2 — Rẽ nhánh tự động chọn kiểm định phù hợp

In [ ]:
if not normal_ok:
    stat, p = stats.mannwhitneyu(g1, g2, alternative="two-sided")
    method = "Mann-Whitney U (dữ liệu không chuẩn)"
elif not equal_var:
    stat, p = stats.ttest_ind(g1, g2, equal_var=False)
    method = "Welch's t-test (phương sai không đồng nhất)"
else:
    stat, p = stats.ttest_ind(g1, g2, equal_var=True)
    method = "Independent-samples t-test (Student)"

print(f"Phương pháp áp dụng: {method}")
print(f"Thống kê kiểm định = {stat:.4f}, p-value = {p:.4f}")
print("=> KẾT LUẬN:", "CÓ khác biệt có ý nghĩa" if p < muc_y_nghia else "KHÔNG có khác biệt có ý nghĩa")

## Bước 3 — Trực quan hoá (đóng băng cùng kết quả trong file audit trail)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.boxplot([g1, g2], tick_labels=groups)
ax.set_ylabel(bien_phu_thuoc)
ax.set_title(f"So sánh {bien_phu_thuoc} theo {bien_nhom}\n({method})")
plt.tight_layout()
plt.show()